In [49]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

df = pd.read_csv("context_data.csv")
df = pd.get_dummies(df, columns=[
    'location','crop','growth_stage','season'
])

print(df.head(4))

  recommendation_label  location_Faisalabad  location_Hyderabad  \
0         pest_control                 True               False   
1           irrigation                 True               False   
2        plant_support                 True               False   
3      soil_management                False               False   

   location_Islamabad  location_Karachi  location_Lahore  location_Multan  \
0               False             False            False            False   
1               False             False            False            False   
2               False             False            False            False   
3               False             False             True            False   

   location_Peshawar  location_Quetta  location_Rawalpindi  ...  crop_Rice  \
0              False            False                False  ...      False   
1              False            False                False  ...      False   
2              False            False       

In [51]:
X = df.drop('recommendation_label', axis=1)
y = df['recommendation_label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [55]:
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score


model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

print("Accuracy:", accuracy_score(y_test, model.predict(X_test)))
print(classification_report(y_test, model.predict(X_test)))

y_pred = model.predict(X_test)

print("\n" + "="*50)
print("MODEL RESULTS")
print("="*50)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.75
               precision    recall  f1-score   support

   irrigation       1.00      1.00      1.00         1
 pest_control       0.67      1.00      0.80         2
plant_support       0.00      0.00      0.00         1

     accuracy                           0.75         4
    macro avg       0.56      0.67      0.60         4
 weighted avg       0.58      0.75      0.65         4


MODEL RESULTS
Accuracy: 0.7500

Classification Report:
               precision    recall  f1-score   support

   irrigation       1.00      1.00      1.00         1
 pest_control       0.67      1.00      0.80         2
plant_support       0.00      0.00      0.00         1

     accuracy                           0.75         4
    macro avg       0.56      0.67      0.60         4
 weighted avg       0.58      0.75      0.65         4



c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [58]:

import numpy as np

def predict_recommendation(location, crop, growth_stage, season):
    # Create a dataframe with the input (same structure as original)
    input_df = pd.DataFrame([{
        'location': location,
        'crop': crop,
        'growth_stage': growth_stage,
        'season': season,
        'recommendation_label': ''  # dummy value
    }])
    
    # Apply same get_dummies transformation
    input_encoded = pd.get_dummies(input_df, columns=[
        'location', 'crop', 'growth_stage', 'season'
    ])
    
    # Drop the dummy recommendation_label column
    if 'recommendation_label' in input_encoded.columns:
        input_encoded = input_encoded.drop('recommendation_label', axis=1)
    
    # Add missing columns (if any)
    for col in X_train.columns:
        if col not in input_encoded.columns:
            input_encoded[col] = 0
    
    # Keep only columns in same order as training
    input_encoded = input_encoded[X_train.columns]
    
    # Predict
    return model.predict(input_encoded)[0]

# Test it
print("="*50)
print("PREDICTIONS")
print("="*50)
print(f"Faisalabad, Pepper, Flowering, Autumn → {predict_recommendation('Faisalabad', 'Pepper', 'Flowering', 'Autumn')}")
print(f"Lahore, Corn, Vegetative, Spring → {predict_recommendation('Lahore', 'Corn', 'Vegetative', 'Spring')}")
print(f"Karachi, Tomato, Flowering, Winter → {predict_recommendation('Karachi', 'Tomato', 'Flowering', 'Winter')}")

PREDICTIONS
Faisalabad, Pepper, Flowering, Autumn → pest_control
Lahore, Corn, Vegetative, Spring → irrigation
Karachi, Tomato, Flowering, Winter → pest_control


In [59]:
# After training, save the feature columns
import joblib

# Save model and feature columns
joblib.dump(model, 'recommendation_model.pkl')
joblib.dump(X_train.columns.tolist(), 'feature_columns.pkl')

['feature_columns.pkl']